# 08 — Avaliar uma arquitetura (rollout real no simulador)

Companheiro do `07_treino_arquitetura.ipynb`: carrega o checkpoint treinado
pra UMA combinação de `BACKBONE_TYPE`+`FUSION_TYPE` e roda a política no
`LiberoEnv` de verdade (não só curva de loss) -- mede taxa de sucesso por
tarefa, com vídeos.

**Diferenças em relação a `05_rollout_hdf5.ipynb`** (que avalia a ablação
de 40 tarefas/ResNet18):
- Usa `configs/libero_object_language.py` (10 tarefas do `libero_object`),
  o mesmo config que `07_treino_arquitetura.ipynb` treina -- `BACKBONE_TYPE`/
  `FUSION_TYPE` resolvem automaticamente o mesmo jeito.
- `cfg["task_suite_name"]` já É a suíte oficial (`"libero_object"`) --
  diferente do config de 40 tarefas (`"libero_40_mixed"`, um pseudo-nome
  só de filtro), não precisa resolver suíte por tarefa via
  `task_mapping_libero40.json`.
- Carrega o modelo via `train_script.build_model_and_optimizer(cfg, device)`
  (mesma função que o treino usa) em vez de reconstruir `ACT(...)` na mão --
  resolve backbone/fusão/texto pareado automaticamente, sem duplicar lógica.
- **Resolução do `LiberoEnv` continua 128×128`** (não 224): os backbones
  `CLIPVisionBackbone`/`SiglipVisionBackbone` já fazem o resize 128->224
  **dentro do próprio `forward()`** (mesmo código do treino) -- manter o
  env em 128 garante que o rollout exercita o MESMO caminho de resize que
  o treino, em vez de pular ele.

`TASK_LIMIT` (seção 3) **precisa bater com o valor usado no treino**
(`07_treino_arquitetura.ipynb`) -- decide tanto quais tarefas são avaliadas
quanto os stats de normalização recalculados (determinísticos dado o
mesmo conjunto de arquivos).

Precisa de `lerobot[libero]` (simulador de verdade) além de `hdf5`/`language`/`vlm`.

## 1. Repositório e ambiente

In [ ]:
!git clone -b teste https://github.com/rafaelheydt/act-lang.git
%cd act-lang

!pip install -q -e ".[hdf5,language,vlm]" "lerobot[libero]"

import sys
sys.path.insert(0, "src")
sys.path.insert(0, ".")
sys.path.insert(0, "scripts")

## 2. Escolha de arquitetura

MESMA escolha usada no treino (`07_treino_arquitetura.ipynb`) -- tem que
bater com o checkpoint que você quer avaliar.

In [ ]:
BACKBONE_TYPE = "clip_vitb32"  # "resnet18" | "clip_vitb32" | "siglip2_base"
FUSION_TYPE = "film"           # "token" | "film" | "cross_attn"

_CLI_CONFIG_NAME = {
    ("resnet18", "token"): "language_token",
    ("resnet18", "film"): "language_film",
    ("resnet18", "cross_attn"): "language_cross_attn",
    ("clip_vitb32", "film"): "language_clip_film",
    ("clip_vitb32", "token"): "language_clip_token",
    ("clip_vitb32", "cross_attn"): "language_clip_cross_attn",
    ("siglip2_base", "film"): "language_siglip_film",
    ("siglip2_base", "token"): "language_siglip_token",
    ("siglip2_base", "cross_attn"): "language_siglip_cross_attn",
}
CLI_CONFIG_NAME = _CLI_CONFIG_NAME[(BACKBONE_TYPE, FUSION_TYPE)]

import train as train_script
cfg = train_script.load_config(CLI_CONFIG_NAME)
print(f"--config {CLI_CONFIG_NAME!r} -> experiment_name={cfg['experiment_name']!r}")
print(f"task_suite_name={cfg['task_suite_name']!r}")

## 3. Normalizadores (mesmo subconjunto de tarefas do treino)

`TASK_LIMIT` precisa bater com o que foi usado em
`07_treino_arquitetura.ipynb` (mesma suíte `libero_object`, mesmo `--limit`) --
decide quais tarefas foram treinadas E reproduz os stats exatos de
normalização (`compute_stats` é determinístico dado o mesmo conjunto de arquivos).

In [ ]:
TASK_LIMIT = 2  # mesmo valor usado no treino -- None se treinou nas 10 completas
DATA_DIR = "/content/libero_hdf5_object"

import subprocess
cmd = [sys.executable, "-u", "scripts/download_libero_hdf5.py", "--out", DATA_DIR, "--suite", "libero_object"]
if TASK_LIMIT is not None:
    cmd += ["--limit", str(TASK_LIMIT)]
print("rodando:", " ".join(cmd))
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
proc.wait()
assert proc.returncode == 0, "download falhou -- veja o log acima"

import torch
from act_lang.data.hdf5_libero import load_meta, build_normalizers_from_meta
from act_lang.utils.runtime import describe_devices, pick_device, is_colab

print(describe_devices())
device = pick_device(preferred_index=cfg.get("device_index"))
print(f"usando: {device}")

meta = load_meta(DATA_DIR)
state_norm, action_norm = build_normalizers_from_meta(meta, device)

from lerobot.datasets.lerobot_dataset import LeRobotDatasetMetadata
lerobot_meta = LeRobotDatasetMetadata("lerobot/libero")
video_fps = int(lerobot_meta.fps)
print(f"normalizadores prontos | video_fps={video_fps}")

## 4. Carregar o checkpoint treinado

`train_script.build_model_and_optimizer` resolve o backbone/fusão/texto
pareado certo a partir do MESMO `cfg` do treino -- mesma função que
`scripts/train.py` usa, sem duplicar lógica de qual text encoder pareia
com qual backbone.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
CHECKPOINT_DIR = Path(f"/content/drive/MyDrive/{cfg['experiment_name']}_hdf5128")
print("checkpoints em:", CHECKPOINT_DIR)

from act_lang.training.checkpoints import load_checkpoint

model, _optimizer = train_script.build_model_and_optimizer(cfg, device)
best = sorted(CHECKPOINT_DIR.glob("best_epoch*.pt"))[-1]  # ou aponte o arquivo à mão
print(f"carregando: {best.name}")
next_epoch, _ = load_checkpoint(best, model, device=device)
model.eval()
print(f"checkpoint da época {next_epoch - 1}")

## 5. Rollout por tarefa

`cfg["task_suite_name"]` (`"libero_object"`) já É a suíte oficial -- monta
o `task_suite` uma vez só, resolve `task_id` por tarefa dentro dela.

In [ ]:
import numpy as np
from libero.libero import benchmark
from lerobot.envs.libero import LiberoEnv
from act_lang.eval.rollout_libero import rollout_libero

all_tasks_sorted = sorted(cfg["task_texts"])
trained_tasks = all_tasks_sorted[:TASK_LIMIT] if TASK_LIMIT is not None else all_tasks_sorted
print(f"{len(trained_tasks)} tarefa(s) treinada(s):")
for t in trained_tasks:
    print(" -", t)

task_suite = benchmark.get_benchmark_dict()[cfg["task_suite_name"]]()
def _task_id(text):
    return next(i for i in range(len(task_suite.tasks)) if task_suite.get_task(i).language == text)

video_root = "/content/libero_rollouts" if is_colab() else str(CHECKPOINT_DIR / "rollouts")

all_results = {}
for task_text in trained_tasks:
    task_id = _task_id(task_text)
    env = LiberoEnv(
        task_suite=task_suite, task_id=task_id, task_suite_name=cfg["task_suite_name"],
        control_mode="relative", render_mode="rgb_array",
        obs_type="pixels_agent_pos",
        observation_width=128, observation_height=128,  # bate com o treino (HDF5 nativo)
    )
    task_video_dir = f"{video_root}/{task_text[:40].replace(' ', '_')}"
    print(f"\n=== {task_text} (task_id={task_id}) ===")
    results = rollout_libero(
        model, env, state_norm, action_norm, device,
        n_episodes=cfg["rollout_n_episodes"], m=cfg["rollout_m"],
        max_steps=cfg["rollout_max_steps"], video_dir=task_video_dir,
        video_fps=video_fps, task_text=task_text,
    )
    env.close()
    all_results[task_text] = results
    sr = np.mean([r["success"] for r in results])
    print(f"taxa de sucesso: {sr:.1%}")

## 6. Resumo

In [ ]:
import pandas as pd

rows = [
    {
        "tarefa": task_text,
        "n_episodios": len(results),
        "taxa_sucesso": np.mean([r["success"] for r in results]),
    }
    for task_text, results in all_results.items()
]
df = pd.DataFrame(rows)
print(df.to_string(index=False))

overall = np.mean([r["success"] for results in all_results.values() for r in results])
print(f"\nTaxa de sucesso geral ({len(all_results)} tarefa(s), {BACKBONE_TYPE}+{FUSION_TYPE}): {overall:.1%}")

## 7. Ver um episódio

In [ ]:
from IPython.display import Video, display

primeira_tarefa, primeiros_resultados = next(iter(all_results.items()))
r0 = primeiros_resultados[0]
tag = "sucesso" if r0["success"] else "falha"
task_video_dir = f"{video_root}/{primeira_tarefa[:40].replace(' ', '_')}"
display(Video(f"{task_video_dir}/ep00_state{r0['init_state_id']:02d}_{tag}.mp4", embed=True, width=700))

## 8. Salvar vídeos no Drive

`video_root` (seção 5) fica no disco local do Colab (`/content`) -- some
quando a sessão cair. Copia tudo pro Drive, junto do checkpoint avaliado.
Seguro rodar de novo depois de mais rollouts (`dirs_exist_ok=True` faz merge
em vez de dar erro por pasta já existir).

In [ ]:
import shutil

destino = CHECKPOINT_DIR / "rollouts"
destino.mkdir(parents=True, exist_ok=True)
shutil.copytree(video_root, destino, dirs_exist_ok=True)

n_videos = len(list(destino.rglob("*.mp4")))
print(f"{n_videos} vídeo(s) copiados de {video_root} para {destino}")

## 9. Exportar resultados

Salva a tabela resumo (CSV) e o resultado bruto por episódio (JSON --
`success`/`steps`/`init_state_id` de cada rollout, útil pra comparar entre
arquiteturas depois) em `CHECKPOINT_DIR/resultados/`, junto do checkpoint
e dos vídeos. Cada rodada gera arquivos novos (timestamp no nome), então
rodar de novo não sobrescreve avaliações anteriores.

In [ ]:
import json
from datetime import datetime

resultados_dir = CHECKPOINT_DIR / "resultados"
resultados_dir.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M")

csv_path = resultados_dir / f"resumo_{timestamp}.csv"
df.to_csv(csv_path, index=False)

json_path = resultados_dir / f"resultados_{timestamp}.json"
payload = {
    "backbone_type": BACKBONE_TYPE,
    "fusion_type": FUSION_TYPE,
    "experiment_name": cfg["experiment_name"],
    "taxa_sucesso_geral": overall,
    "por_tarefa": {task_text: results for task_text, results in all_results.items()},
}
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print(f"resumo (CSV): {csv_path}")
print(f"resultado completo (JSON): {json_path}")